In [1]:
import ibis

con = ibis.postgres.connect(
    user="postgres",
    password="password",
    host="postgres",
    port=5432,
    database="my_db",
)

tbl_name = "air_traffic"


In [2]:
import sys
import os

# Add project root to path
# Get the directory of this file, then go up one level to project root
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.SqlAgent import SqlAgent


In [3]:
base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"
fallback_model = "gpt-5"
temperature = 0
max_token = 10000

In [4]:
agent = SqlAgent(
    api_key=api_key,
    base_url=base_url,
    model=model,
    con=con,
    fallback=True,
    fallback_model=fallback_model,
    tbl_name=tbl_name,
    memory=True,
    memory_size=3,
    enable_logging=True,
    log_to_console=False,
)


In [5]:
question = "How many rows are in the dataset?"

agent.ask_question(question=question, verbose=True)



✓ QUERY SUCCESSFUL

SQL Query:
SELECT
  COUNT(*)
FROM "air_traffic"
LIMIT 10000

Results (1 row):
--------------------------------------------------------------------------------
 count
 38546



QueryOutput(success=True, rows=1, cols=1)

In [6]:
print(agent.chat_history)

Human: How many rows are in the dataset?
AI: SELECT COUNT(*) FROM "air_traffic";


In [7]:
a = agent.ask_question(question=question, verbose=False)
print(a)

QueryOutput(success=True, rows=1, cols=1)


In [8]:
print(a.data)
print(a.query)

   count
0  38546
SELECT COUNT(*) FROM "air_traffic" LIMIT 10000


In [9]:
a.display()


✓ QUERY SUCCESSFUL

SQL Query:
SELECT
  COUNT(*)
FROM "air_traffic"
LIMIT 10000

Results (1 row):
--------------------------------------------------------------------------------
 count
 38546



In [10]:
question = "How many passengers landed during 2024?"
agent.ask_question(question=question, verbose=True)


✓ QUERY SUCCESSFUL

SQL Query:
SELECT
  SUM("Passenger Count")
FROM "air_traffic"
WHERE
  "Activity Type Code" = 'Deplaned' AND EXTRACT(YEAR FROM "Date") = 2024
LIMIT 10000

Results (1 row):
--------------------------------------------------------------------------------
     sum
26079194



QueryOutput(success=True, rows=1, cols=1)

In [11]:
print(agent.chat_history)


Human: How many rows are in the dataset?
AI: SELECT COUNT(*) FROM "air_traffic";
Human: How many rows are in the dataset?
AI: SELECT COUNT(*) FROM "air_traffic";
Human: How many passengers landed during 2024?
AI: SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Activity Type Code" = 'Deplaned' AND EXTRACT(YEAR FROM "Date") = 2024;


In [ ]:
question = "And departure?"
agent.ask_question(question=question, verbose=False)


In [ ]:
print(agent.chat_history)

In [ ]:
question = "How many passengers passed via terminal 1?"
agent.ask_question(question=question, verbose=False)


In [ ]:
print(agent.chat_history)


In [ ]:
question = "That is not the right answer, the values of the Terminal field are 'Terminal 1', 'Terminal 2', etc."
agent.ask_question(question=question, verbose=False)
